# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Chargement des données
Les données `.dat` sont chargées nativement par Pyomo via `model.create_instance(...)` dans la section du modèle.

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = AbstractModel()

## 🔹 Sets

In [ ]:
model.COMPARTIMENTS = Set()
model.CHARGES = Set()
model.ARC = Set(dimen=2, initialize=lambda m: [(i0,i1) for i0 in m.COMPARTIMENTS for i1 in m.CHARGES])

## 🔹 Parameters

In [ ]:
model.cap_poids = Param(model.COMPARTIMENTS, within=NonNegativeReals)
model.cap_volume = Param(model.COMPARTIMENTS, within=NonNegativeReals)
model.poids = Param(model.CHARGES, within=NonNegativeReals)
model.volume = Param(model.CHARGES, within=NonNegativeReals)
model.gain = Param(model.CHARGES, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.COMPARTIMENTS, model.CHARGES, domain=NonNegativeReals)

## 🔹 Data

In [ ]:
model = model.create_instance('../data/Cargo_explicit_test_data.dat')

## 🔹 Constraints

In [ ]:
model.c_for_0 = ConstraintList()
for c in model.COMPARTIMENTS:
    model.c_for_0.add(sum(model.X[c, j] for j in model.CHARGES) <= model.cap_poids[c])
model.c_for_1 = ConstraintList()
for c in model.COMPARTIMENTS:
    model.c_for_1.add(sum(model.volume[j] * model.X[c, j] for j in model.CHARGES) <= model.cap_volume[c])
model.c_for_2 = ConstraintList()
for j in model.CHARGES:
    model.c_for_2.add(sum(model.X[c, j] for c in model.COMPARTIMENTS) <= model.poids[j])
model.c_for_3 = ConstraintList()
for c in model.COMPARTIMENTS:
    model.c_for_3.add(sum(model.cap_poids[u] for u in model.COMPARTIMENTS) * sum(model.X[c, j] for j in model.CHARGES) == model.cap_poids[c] * sum(model.X[t, j] for t,j in model.ARC))

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(sum(( model.gain[j] * model.X[c, j] ) for j in model.CHARGES) for c in model.COMPARTIMENTS), sense=maximize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')